# Train or resume the bubble narration classifier
Use **Godfather**. Edit the first cell, then run all cells. `EPOCHS` is the total target: 40 → 60 runs epochs 41–60. In the same kernel, change `EPOCHS` and rerun the training cell. After a restart, run all cells to restore `last.pt` including optimizer and random states.

`best.pt` preserves the highest validation macro F1, including epoch 0. `last.pt` resumes the last completed epoch. A separate RUN folder is required for changing inputs, train switches or initialization. Old runs without last.pt can supply starting weights but cannot restore optimizer state.

Shared run paths live in config.py under TRAINING_PATHS. Notebook 07 writes deberta_run; notebook 05 reads that source and writes classifier_run; notebook 06 evaluates either run explicitly.


## 1. Choose what to train and which weights to load
DeBERTa stays frozen in this notebook, but its full checkpoint and individual layer overrides are configurable. Use notebook 07 to fine-tune its layers. Set TrainClassifier=False for validation only. ResumeTraining=True automatically uses RUN/last.pt when available.

Full and layer initialization paths apply to new runs. ResumeTraining=True loads the run’s complete last.pt instead; choose a new RUN to mix different checkpoints.

Checkpoint inputs are independent: set DEBERTA_CHECKPOINT_PATH to a full encoder folder or a joint checkpoint file; optionally fill DEBERTA_LAYER_CHECKPOINTS with layer-index/file pairs. CLASSIFIER_CHECKPOINT_PATH controls full classifier loading, while CLASSIFIER_LAYER_CHECKPOINTS overrides named modules. Layer indices are zero-based.


In [16]:
from pathlib import Path
import sys
import json
import random
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, WeightedRandomSampler

ROOT = Path(r'C:/Users/ESHAAN/BYTE/Applied-Task/Magi/FINETUNING-EXPERMENTS/NarrationModelwBubbles')
sys.path.insert(0, str(ROOT))
for name in ['config', 'model', 'bubble_features', 'training_state', 'checkpoint_inputs']:
    sys.modules.pop(name, None)
from config import CONFIG, TRAINING_PATHS, fingerprint, save_json
from model import BoxClassifier, collate, metrics, prediction_rows
from bubble_features import sha256, model_files
from training_state import load_resume, restore_state, save_progress
from checkpoint_inputs import prepare_encoder, frozen_samples, read_weights, load_layer, checkpoint_identity, export_best, check_training_inputs

TrainClassifier = True
TrainDeberta = False
UseClassifierCheckpoint = True
DEBERTA_BASE_PATH = Path(CONFIG['deberta_model'])
SOURCE_RUN = TRAINING_PATHS['deberta_run']
DEBERTA_CHECKPOINT_PATH = SOURCE_RUN / 'deberta'
CLASSIFIER_CHECKPOINT_PATH = SOURCE_RUN / 'classifier.pt'
DEBERTA_LAYER_CHECKPOINTS = {}  # Example: {11: Path(r'C:/my_run/deberta_layers/layer_11.pt')}
CLASSIFIER_LAYER_CHECKPOINTS = {}  # Example: {'classifier': Path(r'C:/my_run/classifier_layers/classifier.pt')}
ResumeTraining = True
RUN = TRAINING_PATHS['classifier_run']
EPOCHS = 60
CLASSIFIER_LR = CONFIG['lr']
TEXT_CHUNK_SIZE = 2

assert not TrainDeberta, 'Use notebook 07 to train DeBERTa.'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])
assert isinstance(EPOCHS, int) and EPOCHS >= 0
saved = load_resume(RUN, ResumeTraining)
print('Resume epoch:', saved['epoch'] if saved else 0, '| target epochs:', EPOCHS)

ENCODER_INPUT_PATH = Path(saved.get('encoder_source', DEBERTA_CHECKPOINT_PATH)) if saved else DEBERTA_CHECKPOINT_PATH
if saved and (RUN / 'deberta/config.json').is_file():
    ENCODER_INPUT_PATH = RUN / 'deberta'
check_training_inputs(DEBERTA_BASE_PATH, ENCODER_INPUT_PATH, CLASSIFIER_CHECKPOINT_PATH,
                      UseClassifierCheckpoint, DEBERTA_LAYER_CHECKPOINTS, CLASSIFIER_LAYER_CHECKPOINTS, saved)
print('Output run:', RUN)


Resume epoch: 40 | target epochs: 60
DeBERTa base: C:\MODELS\deberta-v3-base
DeBERTa checkpoint: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationModelwBubbles\runs_classifier_resume_v1_tuned_Bert\deberta
Output run: C:\Users\ESHAAN\BYTE\Applied-Task\Magi\FINETUNING-EXPERMENTS\NarrationModelwBubbles\runs_classifier_resume_v1_tuned_Bert


## 2. Load cached inputs, the original split and classifier
Uses frozen bubble vectors. Selecting another DeBERTa checkpoint builds and caches new text vectors automatically. Narration-containing sequences retain 2.5× sampling weight; class weights come from original labels. Initialization loads weights only; resuming restores the complete training state.


In [17]:
ENCODER_PATH = prepare_encoder(DEBERTA_BASE_PATH, ENCODER_INPUT_PATH,
                               {} if saved else DEBERTA_LAYER_CHECKPOINTS, CONFIG['cache_dir'])

SAMPLES_PATH = CONFIG['cache_dir'] / 'samples.pt'
if not SAMPLES_PATH.is_file():
    raise FileNotFoundError('Prepare the training caches first using notebooks 01–04.')
data = torch.load(SAMPLES_PATH, map_location='cpu', weights_only=True)
if ENCODER_PATH.resolve() != Path(CONFIG['deberta_model']).resolve():
    data = frozen_samples(data, ENCODER_PATH, CONFIG['cache_dir'], CONFIG['max_tokens'], TEXT_CHUNK_SIZE)
train = [data['samples'][i] for i in data['train_indices'] if (data['samples'][i]['labels'] != -100).any()]
val = [data['samples'][i] for i in data['val_indices']]
visual_dim = train[0]['visual'].shape[-1]
model_args = {key: CONFIG[key] for key in ['dropout', 'num_heads', 'dim_feedforward']}
initial = saved
classifier_weights = saved['state_dict'] if saved is not None else None
if saved is None and UseClassifierCheckpoint:
    classifier_weights, initial = read_weights(CLASSIFIER_CHECKPOINT_PATH, 'classifier')
if initial:
    assert initial.get('visual_dim', visual_dim) == visual_dim
    assert initial.get('max_positions', data['max_positions']) == data['max_positions']
    model_args.update(initial.get('model_args', {}))
classifier = BoxClassifier(visual_dim, data['max_positions'], **model_args).to(DEVICE)
if classifier_weights is not None:
    classifier.load_state_dict(classifier_weights, strict=True)
if saved is None:
    for module_name, path in CLASSIFIER_LAYER_CHECKPOINTS.items():
        load_layer(classifier.get_submodule(module_name), path, module_name, 'classifier')
classifier.requires_grad_(TrainClassifier)
optimizer = torch.optim.AdamW(classifier.parameters(), lr=CLASSIFIER_LR,
                              weight_decay=CONFIG['weight_decay']) if TrainClassifier else None
scaler = None
models = {'classifier': classifier}
weights = [CONFIG['narration_sampling_weight'] if (sample['labels'] == 1).any() else 1.0 for sample in train]
sampler = WeightedRandomSampler(weights, len(train), replacement=True,
                                generator=torch.Generator().manual_seed(CONFIG['seed']))
train_loader = DataLoader(train, batch_size=CONFIG['batch_size'], sampler=sampler, collate_fn=collate)
val_loader = DataLoader(val, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate)
criterion = nn.CrossEntropyLoss(weight=data['class_weights'].to(DEVICE), ignore_index=-100, reduction='sum')

feature_contract = {
    'name': 'NarrationModelwBubbles', 'settings': CONFIG['bubble_settings'],
    'helper_sha256': sha256(ROOT / 'bubble_features.py'),
    'ocr_selector_sha256': sha256(ROOT / 'ocr_selection.py'),
    'bubble_model_sha256': sha256(CONFIG['bubble_settings']['bubble_model']),
    'magi_files': model_files(Path(CONFIG['magi_model'])),
    'deberta_files': model_files(ENCODER_PATH),
    'max_tokens': CONFIG['max_tokens'],
}

contract = fingerprint({
    'samples': data['signature'], 'features': feature_contract, 'model_args': model_args,
    'train_classifier': TrainClassifier, 'train_deberta': TrainDeberta,
    'classifier_lr': CLASSIFIER_LR, 'batch_size': CONFIG['batch_size'],
    'weight_decay': CONFIG['weight_decay'], 'sampling': CONFIG['narration_sampling_weight'],
    'seed': CONFIG['seed'], 'code': sha256(ROOT / 'training_state.py'),
})
if CLASSIFIER_LAYER_CHECKPOINTS:
    contract = fingerprint({'base_contract': contract, 'classifier_layers': {
        key: checkpoint_identity(path) for key, path in CLASSIFIER_LAYER_CHECKPOINTS.items()}})
metadata = {
    'training_contract': contract, 'samples_signature': data['signature'],
    'visual_dim': visual_dim, 'max_positions': data['max_positions'],
    'model_args': model_args, 'class_weights': data['class_weights'],
    'feature_contract': feature_contract,
    'config': json.loads(json.dumps(CONFIG, default=str)),
    'encoder_source': str(ENCODER_PATH),
}
print('Train / validation:', len(train), len(val), '| classifier trainable:', TrainClassifier)

del initial, classifier_weights


Train / validation: 63 16 | classifier trainable: True


## 3. Training and validation helpers
Ignored boxes remain context but do not contribute loss. Validation has no updates or oversampling.


In [18]:
def run_epoch(loader, training):
    classifier.train(training and TrainClassifier)
    total_loss, labeled_count, rows = 0.0, 0, []
    for batch in loader:
        batch = {key: value.to(DEVICE) if torch.is_tensor(value) else value for key, value in batch.items()}
        count = int((batch['labels'] != -100).sum())
        with torch.set_grad_enabled(training and TrainClassifier):
            logits = classifier(batch)
            loss = criterion(logits.reshape(-1, 2), batch['labels'].reshape(-1))
            if not torch.isfinite(loss):
                raise FloatingPointError('Nonfinite loss.')
            if training and TrainClassifier and count:
                optimizer.zero_grad(set_to_none=True)
                (loss / count).backward()
                torch.nn.utils.clip_grad_norm_(classifier.parameters(), 1.0)
                optimizer.step()
        total_loss += float(loss.detach())
        labeled_count += count
        rows.extend(prediction_rows(batch, logits.detach().float().softmax(-1).cpu()))
    result = metrics(rows)
    result['loss'] = total_loss / labeled_count if labeled_count else None
    return result, rows


## 4. Train to the total EPOCHS target
Edit EPOCHS above and rerun this cell. Completed epochs are skipped. Every epoch saves last.pt; best.pt changes only on improved validation macro F1. No automatic early stopping prevents extending the target.


In [19]:
# Reload last.pt every time this cell runs, including after a kernel restart.
saved = load_resume(RUN, ResumeTraining)
if TrainClassifier or TrainDeberta:
    restore_state(saved, models, optimizer, scaler, sampler, contract)
elif saved is not None:
    assert saved['samples_signature'] == data['signature'], 'Evaluation inputs changed.'
    for name, model in models.items():
        model.load_state_dict(saved['models'][name], strict=True)
history = list(saved['history']) if saved else []
completed = saved['epoch'] if saved else 0
best_f1 = saved['best_f1'] if saved else -1.0
best_epoch = saved['best_epoch'] if saved else 0

if not (TrainClassifier or TrainDeberta):
    validation, _ = run_epoch(val_loader, False)
    print('Evaluation only:', json.dumps(validation, indent=2))
else:
    if saved is None:
        baseline, rows = run_epoch(val_loader, False)
        best_f1 = baseline['macro_f1']
        metadata['baseline_metrics'] = baseline
        save_progress(RUN, models, optimizer, scaler, sampler, metadata,
                      0, history, best_f1, 0, baseline, True)
        save_json(RUN / 'best_validation_predictions.json', rows)
        export_best(RUN, models, metadata, ENCODER_PATH)
        print('Epoch 0 validation:', json.dumps(baseline, indent=2))
    else:
        metadata['baseline_metrics'] = saved['baseline_metrics']
    saved = None
    for epoch in range(completed + 1, EPOCHS + 1):
        print(f'Epoch {epoch}/{EPOCHS}', flush=True)
        train_metrics, _ = run_epoch(train_loader, True)
        val_metrics, rows = run_epoch(val_loader, False)
        history.append({'epoch': epoch, 'train': train_metrics, 'val': val_metrics})
        improved = val_metrics['macro_f1'] > best_f1
        if improved:
            best_f1, best_epoch = val_metrics['macro_f1'], epoch
        save_progress(RUN, models, optimizer, scaler, sampler, metadata,
                      epoch, history, best_f1, best_epoch, val_metrics, improved)
        save_json(RUN / f'val_predictions_epoch_{epoch:02d}.json', rows)
        if improved:
            save_json(RUN / 'best_validation_predictions.json', rows)
            export_best(RUN, models, metadata, ENCODER_PATH)
        completed = epoch
        print(json.dumps(history[-1], indent=2), flush=True)
    print('Completed:', completed, '| best epoch:', best_epoch, '| best macro F1:', best_f1)
    print('Increase EPOCHS and rerun this cell to continue.')


Epoch 41/60
{
  "epoch": 41,
  "train": {
    "macro_f1": 0.9793855042016807,
    "balanced_accuracy": 0.99220856249326,
    "SPOKEN": {
      "precision": 0.9991039426523297,
      "recall": 0.9919928825622776,
      "f1": 0.9955357142857143,
      "support": 1124,
      "p_true_mean": 0.9843530525544325,
      "p_true_median": 0.9941279888153076,
      "p_true_gt_0.9": 0.9902135231316725,
      "p_true_lt_0.5": 0.00800711743772242
    },
    "NARRATED": {
      "precision": 0.9357142857142857,
      "recall": 0.9924242424242424,
      "f1": 0.9632352941176471,
      "support": 132,
      "p_true_mean": 0.9900652012561985,
      "p_true_median": 0.9976412951946259,
      "p_true_gt_0.9": 0.9924242424242424,
      "p_true_lt_0.5": 0.007575757575757576
    },
    "loss": 0.01188839984772976
  },
  "val": {
    "macro_f1": 0.8642192280153616,
    "balanced_accuracy": 0.8775978407557354,
    "SPOKEN": {
      "precision": 0.9894366197183099,
      "recall": 0.9859649122807017,
      "f1":

## 5. Inspect the preserved validation history


In [20]:
import pandas as pd

if (RUN / 'last.pt').is_file():
    progress = torch.load(RUN / 'last.pt', map_location='cpu', weights_only=True)
    display(pd.DataFrame([
        {'epoch': row['epoch'], 'train_loss': row['train']['loss'], 'val_loss': row['val']['loss'],
         'macro_f1': row['val']['macro_f1'],
         'narration_precision': row['val']['NARRATED']['precision'],
         'narration_recall': row['val']['NARRATED']['recall'],
         'narration_f1': row['val']['NARRATED']['f1']}
        for row in progress['history']
    ]))
    print('Best epoch:', progress['best_epoch'], '| best macro F1:', progress['best_f1'])
    del progress


,epoch,train_loss,val_loss,macro_f1,narration_precision,narration_recall,narration_f1
0,1,0.193192,0.107275,0.696096,0.454545,0.384615,0.416667
1,2,0.042010,0.185764,0.716820,0.555556,0.384615,0.454545
2,3,0.049732,0.122677,0.665078,0.264706,0.692308,0.382979
3,4,0.041676,0.254767,0.700994,0.666667,0.307692,0.421053
4,5,0.020775,0.314333,0.679172,0.500000,0.307692,0.380952
5,6,0.008864,0.369079,0.716820,0.555556,0.384615,0.454545
6,7,0.010956,0.599172,0.488851,0.000000,0.000000,0.000000
7,8,0.025169,0.497928,0.561119,1.000000,0.076923,0.142857
8,9,0.011129,0.113483,0.802699,0.500000,0.846154,0.628571
9,10,0.013899,0.134849,0.781662,0.476190,0.769231,0.588235


Best epoch: 40 | best macro F1: 0.9195681511470986


## 6. Use the best classifier in the full pipeline
Set these paths in the inference notebook; text embeddings use the selected frozen DeBERTa checkpoint recorded in inference_config.py.


In [21]:
print('NARRATION_MODEL_PATH =', repr(str(ROOT / 'model.py')))
print('NARRATION_CHECKPOINT_PATH =', repr(str(RUN / 'best.pt')))
print('NARRATION_CONFIG_PATH =', repr(str(RUN / 'inference_config.py')))


NARRATION_MODEL_PATH = 'C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\Magi\\FINETUNING-EXPERMENTS\\NarrationModelwBubbles\\model.py'
NARRATION_CHECKPOINT_PATH = 'C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\Magi\\FINETUNING-EXPERMENTS\\NarrationModelwBubbles\\runs_classifier_resume_v1_tuned_Bert\\best.pt'
NARRATION_CONFIG_PATH = 'C:\\Users\\ESHAAN\\BYTE\\Applied-Task\\Magi\\FINETUNING-EXPERMENTS\\NarrationModelwBubbles\\runs_classifier_resume_v1_tuned_Bert\\inference_config.py'
